# Revisión PROFILING (calidad del dato crudo)

Evaluación de las 8 dimensiones de calidad sobre cada archivo bronce
(accuracy, completeness, consistency, integrity, reasonableness, timeliness,
uniqueness, validity). Es **solo lectura**: documenta el estado del crudo y
justifica las reglas de limpieza de silver.

In [1]:
from pathlib import Path

import polars as pl
import pyarrow.parquet as pq

DATA = Path("../data") if Path("../data").exists() else Path("data")
CATS = ["green", "yellow", "fhv", "fhvhv"]
YEARS = [2023, 2024, 2025]
MESES = [(y, m) for y in YEARS for m in range(1, 13)]
pl.Config.set_tbl_rows(80)
pl.Config.set_tbl_cols(25)


def parquet_rows(path: Path) -> int | None:
    """Filas totales de un archivo/directorio parquet leyendo solo metadatos.

    Ojo: Spark escribe "archivos" mensuales como DIRECTORIOS llamados *.parquet
    con part-files dentro — por eso se filtra con is_file(). Se ignora la
    basura _temporary y los archivos de 0 bytes que deja un job interrumpido.
    """
    if not path.exists():
        return None
    files = (
        [path]
        if path.is_file()
        else [
            f for f in path.rglob("*.parquet")
            if f.is_file() and f.stat().st_size > 0 and "_temporary" not in str(f)
        ]
    )
    if not files:
        return None
    return sum(pq.ParquetFile(f).metadata.num_rows for f in files)


def dir_mb(path: Path) -> float:
    """Peso en MB de un archivo o directorio."""
    if not path.exists():
        return 0.0
    files = [path] if path.is_file() else [f for f in path.rglob("*") if f.is_file()]
    return round(sum(f.stat().st_size for f in files) / 1024**2, 2)

In [2]:
import json

filas = []
for f in sorted((DATA / "profiling").rglob("*.json")):
    r = json.loads(f.read_text(encoding="utf-8"))
    for d in r["dimensions"]:
        filas.append({
            "dataset": r["meta"]["name"],
            "categoria": r["meta"]["category"],
            "dimension": d["dimension"],
            "score": d["score"],
            "aprobada": d["passed"],
            "score_global": r["overall_score"],
        })
prof = pl.DataFrame(filas)
print(f"{prof['dataset'].n_unique()} datasets perfilados x {prof['dimension'].n_unique()} dimensiones")

144 datasets perfilados x 8 dimensiones


## Score promedio por dimensión y categoría

Scores < 1.0 señalan problemas en el crudo que silver corrige o rechaza.

In [3]:
(prof.group_by("categoria", "dimension")
     .agg(pl.col("score").mean().round(4).alias("score_promedio"))
     .pivot(values="score_promedio", index="dimension", on="categoria")
     .sort("dimension"))

dimension,fhv,fhvhv,yellow,green
str,f64,f64,f64,f64
"""accuracy""",1.0,0.0007,0.6422,0.8534
"""completeness""",0.8619,1.0,0.975,0.9843
"""consistency""",0.9999,1.0,1.0,0.9999
"""integrity""",1.0,1.0,1.0,1.0
"""reasonableness""",1.0,0.993,0.9957,0.9994
"""timeliness""",1.0,1.0,1.0,0.9997
"""uniqueness""",0.9959,1.0,0.9723,0.9939
"""validity""",0.0,0.0,0.0,0.0


## Dimensiones reprobadas (dónde el crudo viene sucio)

In [4]:
(prof.filter(~pl.col("aprobada"))
     .group_by("categoria", "dimension")
     .agg(pl.len().alias("meses_reprobados"), pl.col("score").min().round(4).alias("peor_score"))
     .sort("categoria", "dimension"))

categoria,dimension,meses_reprobados,peor_score
str,str,u32,f64
"""fhv""","""completeness""",36,0.8486
"""fhv""","""consistency""",36,0.9995
"""fhv""","""uniqueness""",36,0.9926
"""fhv""","""validity""",36,0.0
"""fhvhv""","""accuracy""",36,0.0004
"""fhvhv""","""consistency""",4,0.9999
"""fhvhv""","""reasonableness""",36,0.9911
"""fhvhv""","""uniqueness""",36,0.9997
"""fhvhv""","""validity""",36,0.0


## Los 10 peores dataset × dimensión

In [5]:
prof.sort("score").head(10).select("dataset", "dimension", "score", "aprobada")

dataset,dimension,score,aprobada
str,str,f64,bool
"""fhv_2023-01""","""validity""",0.0,false
"""fhv_2023-02""","""validity""",0.0,false
"""fhv_2023-03""","""validity""",0.0,false
"""fhv_2023-04""","""validity""",0.0,false
"""fhv_2023-05""","""validity""",0.0,false
"""fhv_2023-06""","""validity""",0.0,false
"""fhv_2023-07""","""validity""",0.0,false
"""fhv_2023-08""","""validity""",0.0,false
"""fhv_2023-09""","""validity""",0.0,false
